In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# データ操作と数値計算
import pandas as pd
import numpy as np
import re # 正規表現用

# 自然言語処理 (NLP) 関連
import nltk
from nltk.corpus import stopwords # ストップワード除去用
from nltk.tokenize import word_tokenize # 単語分割用
# NLTKデータのダウンロード（必要であればコメントを外して実行）
# nltk.download('stopwords')
# nltk.download('punkt')

# Hugging Face Transformers と PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW # AdamWはtorch.optimから直接インポート
from transformers import get_linear_schedule_with_warmup

# モデル評価とデータ分割
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# 可視化ライブラリ
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# その他のユーティリティ
import os # ファイルパス操作用

import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW 
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score # 評価指標計算用

In [ ]:
print("--- データの読み込みと初期確認 ---")
# データセットのパスを定義
train_path = '/kaggle/input/nlp-getting-started/train.csv'
test_path = '/kaggle/input/nlp-getting-started/test.csv'
sample_submission_path = '/kaggle/input/nlp-getting-started/sample_submission.csv'

# CSVファイルをPandas DataFrameとして読み込む
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

# データの形状と最初の数行を表示して確認
print("\n訓練データ (train_df) の情報:")
print(train_df.shape)
print(train_df.head())

print("\nテストデータ (test_df) の情報:")
print(test_df.shape)
print(test_df.head())

# 欠損値の確認
print("\n訓練データの欠損値:")
print(train_df.isnull().sum())
print("\nテストデータの欠損値:")
print(test_df.isnull().sum())

In [ ]:
print("\n--- 欠損値の処理 ---")
# 'keyword'と'location'カラムの欠損値を'no_keyword'/'no_location'で埋める
# BERTはテキストの文脈を重視するため、これらの情報は直接利用しないが、データの一貫性を保つため処理
train_df['keyword'] = train_df['keyword'].fillna('no_keyword')
test_df['keyword'] = test_df['keyword'].fillna('no_keyword')
train_df['location'] = train_df['location'].fillna('no_location')
test_df['location'] = test_df['location'].fillna('no_location')

In [ ]:
print("\n--- BERT向けテキストクリーニング ---")
# BERTは前処理に非常に強いが、URLやHTMLタグ、絵文字は意味を歪める可能性があるため除去
def clean_text_for_bert(text):
    # URLを除去
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # HTMLタグを除去
    text = re.sub(r'<.*?>', '', text)
    # 絵文字を除去
    emoji_pattern = re.compile(
        '['
        '\U0001F600-\U0001F64F' '\U0001F300-\U0001F5FF'
        '\U0001F680-\U0001F6FF' '\U0001F1E0-\U0001F1FF'
        '\U00002702-\U000027B0' '\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    # BERTは小文字化を内部で行うため、ここでは明示的に行わない
    # BERTは句読点や数字もトークンとして扱うので、ここでは除去しない
    return text

# クリーニング関数を適用
train_df['cleaned_text_bert'] = train_df['text'].apply(clean_text_for_bert)
test_df['cleaned_text_bert'] = test_df['text'].apply(clean_text_for_bert)

# # クリーニング後のテキスト例を確認
# print(f"クリーニング前のテキスト例: {train_df['text'].iloc[0]}")
# print(f"クリーニング後のテキスト例: {train_df['cleaned_text_bert'].iloc[0]}")

In [ ]:
# ターゲット変数とクリーニング済みテキストデータを取得
sentences = train_df['cleaned_text_bert'].values
labels = train_df['target'].values
test_sentences = test_df['cleaned_text_bert'].values

print(f"\n訓練データのツイート数: {len(sentences)}")
print(f"テストデータのツイート数: {len(test_sentences)}")
print(f"最初の訓練ツイートの例: {sentences[0]}")

In [ ]:
print("\n--- 仮説検証のための可視化を開始します ---")

# ターゲット変数ごとのキーワード出現頻度
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['keyword'] != 'no_keyword'], y='keyword', hue='target',
              order=train_df[train_df['keyword'] != 'no_keyword']['keyword'].value_counts().index[:20])
plt.title('Keyword Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Keyword')
plt.tight_layout()
plt.show()

# ワードクラウドの生成
stop_words = set(stopwords.words('english'))

def generate_wordcloud(text_data, title):
    # テキストを結合し、小文字化、単語分割、ストップワード除去
    all_words = ' '.join(text_data).lower()
    words = word_tokenize(all_words)
    filtered_words = [word for word in words if word.isalnum() and word not in stop_words]
    cleaned_text = ' '.join(filtered_words)

    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          collocations=False).generate(cleaned_text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

# 災害ツイートのワードクラウド
generate_wordcloud(train_df[train_df['target'] == 1]['cleaned_text_bert'], 'Word Cloud for Disaster Tweets')

# 非災害ツイートのワードクラウド
generate_wordcloud(train_df[train_df['target'] == 0]['cleaned_text_bert'], 'Word Cloud for Non-Disaster Tweets')


# ターゲット変数ごとのロケーション分布 (上位20)
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['location'] != 'no_location'], y='location', hue='target',
              order=train_df[train_df['location'] != 'no_location']['location'].value_counts().index[:20])
plt.title('Location Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Location')
plt.tight_layout()
plt.show()

# 欠損値の分布 (keywordとlocation)
# 欠損値の有無を示す新しいカラムを作成
train_df['keyword_missing'] = train_df['keyword'] == 'no_keyword'
train_df['location_missing'] = train_df['location'] == 'no_location'

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(data=train_df, x='keyword_missing', hue='target')
plt.title('Relationship between Keyword Missingness and Target')
plt.xlabel('Keyword Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.subplot(1, 2, 2)
sns.countplot(data=train_df, x='location_missing', hue='target')
plt.title('Relationship between Location Missingness and Target')
plt.xlabel('Location Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.tight_layout()
plt.show()

print("\n--- 仮説検証のための可視化が完了しました ---")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# --- トークン数とtargetの関係の可視化 ---
print("\n--- トークン数とtargetの関係の可視化 ---")

# 事前学習済みモデルのトークナイザーをロード
# あなたのコードで使われているRoBERTaモデルの名前に変更してください。
# 例: 'roberta-base', 'microsoft/codebert-base-mlm' など
tokenizer_name = 'roberta-base' # <--- ここをあなたのモデル名に修正！
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# 訓練データにおける各ツイートのトークン数を計算
# 'text'カラムが存在すると仮定します。もし別のカラム名であれば適宜変更してください。
if 'text' in train_df.columns: # 'text'カラムの存在をチェック
    train_df['num_tokens'] = train_df['text'].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))
    print(f"訓練データのトークン数の平均: {train_df['num_tokens'].mean():.2f}")
    print(f"訓練データのトークン数の最大: {train_df['num_tokens'].max()}")
    print(f"訓練データのトークン数の最小: {train_df['num_tokens'].min()}")
else:
    print("警告: 'text' カラムが訓練データに見つかりませんでした。トークン数の計算をスキップします。")


# targetごとのトークン数の分布をヒストグラムで表示
if 'text' in train_df.columns and 'target' in train_df.columns: # 'text'と'target'カラムの存在をチェック
    plt.figure(figsize=(12, 6))
    sns.histplot(data=train_df, x='num_tokens', hue='target', multiple='stack', bins=50)
    plt.title('Distribution of Token Counts by Target')
    plt.xlabel('Number of Tokens')
    plt.ylabel('Count')
    plt.show()

    # targetとトークン数の関係を箱ひげ図で表示
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=train_df, x='target', y='num_tokens')
    plt.title('Token Counts per Target Category')
    plt.xlabel('Target')
    plt.ylabel('Number of Tokens')
    plt.show()
else:
    print("警告: 'text' または 'target' カラムが訓練データに見つかりませんでした。可視化をスキップします。")

# テストデータにおける各ツイートのトークン数を計算（参考情報として）
if 'text' in test_df.columns: # 'text'カラムの存在をチェック
    test_df['num_tokens'] = test_df['text'].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))
    print(f"テストデータのトークン数の平均: {test_df['num_tokens'].mean():.2f}")
    print(f"テストデータのトークン数の最大: {test_df['num_tokens'].max()}")
    print(f"テストデータのトークン数の最小: {test_df['num_tokens'].min()}")
else:
    print("警告: 'text' カラムがテストデータに見つかりませんでした。テストデータのトークン数の計算をスキップします。")

# --- 可視化コードの終わり ---

In [ ]:
# --- RoBERTaトークナイザーとモデルの初期化 (既存のBERT初期化と置き換え) ---

# トークナイザーのロード
print("--- RoBERTaTokenizerをロードします ---")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base', do_lower_case=True)

# エンコーディング関数 (RoBERTaTokenizerを使用するように変更)
def encode_texts(tokenizer, texts, max_len):
    input_ids = []
    attention_masks = []
    for text in texts:
        encoded_dict = tokenizer.encode_plus(
            text,
            add_special_tokens=True, # [CLS], [SEP]を追加
            max_length=max_len,      # 最大長までパディングまたは切り捨て
            padding='max_length',    # max_lengthまでパディング
            truncation=True,         # max_lengthより長い場合は切り捨て
            return_attention_mask=True,
            return_tensors='pt',     # PyTorchテンソルで返す
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
    return torch.cat(input_ids, dim=0), torch.cat(attention_masks, dim=0)

MAX_LEN = 128 # 最大トークン長 (既存のものを使用するか、ここで定義)

print(f"--- 訓練データをエンコードします (MAX_LEN={MAX_LEN}) ---")
train_inputs, train_masks = encode_texts(tokenizer, sentences, MAX_LEN)
test_inputs, test_masks = encode_texts(tokenizer, test_sentences, MAX_LEN)

train_labels = torch.tensor(labels)


# --- 訓練・検証データセットの分割 (既存コードをそのまま利用) ---
# train_inputs, train_masks, train_labels, test_inputs, test_masks, labels変数が上記で準備できた後に続きます
X_train, X_val, y_train, y_val = train_test_split(
    train_inputs, train_labels, random_state=42, test_size=0.1
)
train_masks, val_masks, _, _ = train_test_split(
    train_masks, train_labels, random_state=42, test_size=0.1
)

# --- データローダーの準備 (既存コードをそのまま利用) ---
batch_size = 32 # バッチサイズ (既存コードと同じ)
train_dataset = TensorDataset(X_train, train_masks, y_train)
val_dataset = TensorDataset(X_val, val_masks, y_val)
train_dataloader = DataLoader(
    train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size
)
validation_dataloader = DataLoader(
    val_dataset, sampler=SequentialSampler(val_dataset), batch_size=batch_size
)
# テストデータローダーの準備 (予測フェーズで使用)
test_dataset = TensorDataset(test_inputs, test_masks)
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=batch_size)


# --- RoBERTaモデルのロード (既存のBERTモデルロードを置き換え) ---
print("--- RobertaForSequenceClassificationモデルをロードします ---")
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',           # 事前学習済みRoBERTaモデル
    num_labels=2,             # 出力ラベルの数 (災害/非災害)
    output_attentions=False,
    output_hidden_states=False,
)




In [ ]:
# GPUが利用可能かチェックし、モデルをGPUへ 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"--- モデルを {device} に移動しました ---")

# --- オプティマイザと学習率スケジューラ  ---
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 3 # エポック数 
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)
print("\n--- 訓練を開始します ---")
for epoch_i in range(0, epochs):
    # 訓練ステップ
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        # ここを修正: token_type_ids=None にするか、引数自体を削除
        outputs = model(b_input_ids,
                        token_type_ids=None, # RoBERTaはtoken_type_idsを使いません
                        attention_mask=b_input_mask,
                        labels=b_labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # 勾配クリッピング
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"  Average training loss: {avg_train_loss:.2f}")

    # 検証ステップ
    print("\n--- 検証を開始します ---")
    model.eval()
    total_eval_accuracy = 0
    total_eval_f1 = 0
    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(b_input_ids,
                            token_type_ids=None, # RoBERTaはtoken_type_idsを使いません
                            attention_mask=b_input_mask,
                            labels=b_labels)
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        preds = np.argmax(logits, axis=1).flatten()
        total_eval_accuracy += accuracy_score(label_ids, preds)
        total_eval_f1 += f1_score(label_ids, preds, average='weighted')

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_f1 = total_eval_f1 / len(validation_dataloader)
    print(f"  検証Accuracy: {avg_val_accuracy:.2f}")
    print(f"  検証F1スコア: {avg_val_f1:.2f}")
print("\n--- 訓練が完了しました ---")

In [ ]:
print("\n--- テストデータでの予測を開始します ---")
model.eval() # モデルを評価モードに設定

predictions = [] # 予測結果を格納するリスト

# テストデータローダーからバッチごとにデータを取得
for batch in test_dataloader:
    # バッチデータをGPUに移動
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)

    # 勾配計算を無効化
    with torch.no_grad():
        # モデルにデータを入力 (ラベルは不要)
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs.logits # モデルの出力 (ロジット)
    # ロジットをCPUに移動し、Numpy配列に変換
    logits = logits.detach().cpu().numpy()
    predictions.append(logits)

# 予測結果を連結し、最も高いlogitを持つクラスを選択
# np.concatenateでリスト内のNumpy配列を結合
# np.argmaxで各行（ツイート）で最も高い値を持つインデックス（クラスID）を取得
final_predictions_bert = np.argmax(np.concatenate(predictions, axis=0), axis=1)

print(f"最終予測結果の形状: {final_predictions_bert.shape}")
print(f"最終予測結果の最初の5件: {final_predictions_bert[:5]}")

print("\n--- 提出用ファイルの作成と保存 ---")
# 提出用DataFrameを作成
submission_df_bert = sample_submission_df.copy()
submission_df_bert['target'] = final_predictions_bert # 予測結果を'target'カラムに代入

# 提出ファイルの最初の数行を表示して確認
print("\n生成された提出ファイルのプレビュー:")
print(submission_df_bert.head())

# 提出ファイルをCSV形式で保存
# index=Falseとすることで、DataFrameのインデックスがCSVに書き込まれないようにする
submission_df_bert.to_csv('submission.csv', index=False)

print("\nsubmission.csv が正常に生成され、/kaggle/working/ ディレクトリに保存されました。")
print("このファイルをKaggleに提出してください。")